## Local Store Synthetic Data Generator

Generates synthetic data about local stores in a given city, seeded from an example JSON structure.

Defaults to a local **Ollama `llama3.1:8b`** model, with a dropdown to switch to **OpenAI**, **Anthropic**, or **Google Gemini** instead.

In [2]:
import json
import gradio as gr
from llm_utils import load_api_keys, get_llm_client

In [3]:
# Prints which provider keys are available in .env. None of these are required
# for the default local Ollama model - they're only needed if you pick that provider.
load_api_keys()

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)
Grok API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)


In [4]:
import os

# All providers are called through the OpenAI-compatible chat completions API,
# just with a different base_url + api_key + model per provider.
MODEL_REGISTRY = {
    "Ollama - llama3.1:8b (local)": {
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama",
        "model": "llama3.1:8b",
    },
    "OpenAI - gpt-4o-mini": {
        "base_url": None,
        "api_key": os.getenv("OPENAI_API_KEY"),
        "model": "gpt-4o-mini",
    },
    "Anthropic - claude-sonnet-4-5": {
        "base_url": "https://api.anthropic.com/v1/",
        "api_key": os.getenv("ANTHROPIC_API_KEY"),
        "model": "claude-sonnet-4-5-20250929",
    },
    "Google - gemini-2.0-flash": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "api_key": os.getenv("GOOGLE_API_KEY"),
        "model": "gemini-2.0-flash",
    },
}

DEFAULT_MODEL = "Ollama - llama3.1:8b (local)"

In [9]:
EXAMPLE_STORES = [
    {"name": "Honest Green", "address": "Av. Colon 233, Valencia", "category": "Restaurant"},
    {"name": "EcoCycle", "address": "Calle del Sol 123, Madrid", "category": "Hardware Store"},
    {"name": "Bloom Market", "address": "Plaza Mayor 456, Barcelona", "category": "Grocery Store"},
    {"name": "Green Earth", "address": "Paseo de la Castellana 789, Valencia", "category": "Furniture Store"},
    {"name": "Sunflower Foods", "address": "Calle de la Paz 101, Seville", "category": "Food Store"},
    {"name": "EcoStore", "address": "Av. de la Constitucion 202, Madrid", "category": "Clothing Store"},
    {"name": "Nature's Nook", "address": "Calle del Arte 456, Granada", "category": "Home Goods"},
    {"name": "Fresh Pick", "address": "Plaza del Ayuntamiento 789, Valencia", "category": "Fruit and Vegetable Store"},
    {"name": "Recycle Zone", "address": "Calle de la Esperanza 101, Seville", "category": "Recycling Center"},
    {"name": "Green Home", "address": "Av. del Pilar 202, Barcelona", "category": "Home Decor"},
]

DEFAULT_EXAMPLE_JSON = json.dumps(EXAMPLE_STORES, indent=2)

In [10]:
def build_prompt(city, sample_size, example_data):
    return f"""
  You are an expert synthetic dataset generator specializing in local businesses.

  City: {city}

  Here is example data showing the structure and style of records to produce:
  {example_data}

  Generate {sample_size} new, realistic but fictional local stores located in {city}.
  Vary the store categories (restaurants, grocery stores, hardware stores, clothing
  stores, home goods, recycling centers, etc.) and use plausible street names and
  addresses for {city}.

  Return ONLY a valid JSON array of records following the same structure as the example.
  The output must start with "[" and end with "]".
  Do not include backticks.
  Do not include explanations.
  Do not include markdown.
  Do not include comments.
"""

In [11]:
def generate_with_model(prompt, model_key):
    config = MODEL_REGISTRY[model_key]

    if not config["api_key"]:
        raise ValueError(f"No API key configured for '{model_key}'. Add the relevant key to your .env file.")

    client = get_llm_client(api_key=config["api_key"], base_url=config["base_url"])
    response = client.chat.completions.create(
        model=config["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()

In [12]:
def validate_json(output_text):
    try:
        data = json.loads(output_text)
        return data, None
    except Exception as e:
        return None, str(e)

In [13]:
def generate_dataset(city, example_data, sample_size, model_key):
    prompt = build_prompt(city, sample_size, example_data)
    try:
        raw_output = generate_with_model(prompt, model_key)
        data, error = validate_json(raw_output)
        if error:
            return f"JSON Error: {error}", raw_output

        return "Generated successfully", json.dumps(data, indent=2)
    except Exception as e:
        return f"Error: {str(e)}", ""

In [14]:
with gr.Blocks() as demo:
    gr.Markdown("## Local Store Synthetic Data Generator")

    city = gr.Textbox(label="City", value="Valencia")
    example_data = gr.Textbox(label="Example Data (JSON)", value=DEFAULT_EXAMPLE_JSON, lines=10)

    model_choice = gr.Dropdown(
        choices=list(MODEL_REGISTRY.keys()),
        value=DEFAULT_MODEL,
        label="Select Model",
    )
    sample_size = gr.Slider(1, 50, value=10, step=1, label="Sample Size")

    status = gr.Textbox(label="Status")
    output_display = gr.Code(label="Generated JSON output", language="json")
    generate_btn = gr.Button("Generate dataset")

    generate_btn.click(
        generate_dataset,
        inputs=[city, example_data, sample_size, model_choice],
        outputs=[status, output_display],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
